In [8]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 1. LOAD TELCO CHURN DIRECTLY FROM OPENML (Zero 404 or missing library errors)
print("Fetching Telco Churn dataset...")
data = fetch_openml(data_id=42178, as_frame=True, parser="auto")
df = data.frame

print("Dataset Loaded Successfully! Shape:", df.shape)

# 2. DATA CLEANING & PREPROCESSING
# Identify target column (Churn / churn)
target_col = [c for c in df.columns if "churn" in c.lower()][0]

# Clean TotalCharges if present
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

if "customerID" in df.columns:
    df.drop(columns=["customerID"], inplace=True)

# Map target variable to 0 and 1
df[target_col] = (
    df[target_col].astype(str).str.lower().map({"yes": 1, "no": 0, "1": 1, "0": 0})
)

# Select features
cat_cols = (
    df.drop(columns=[target_col]).select_dtypes(include=["object", "category"]).columns.tolist()
)
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
if target_col in num_cols:
    num_cols.remove(target_col)

# One-hot encoding
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col].astype(int)

# Train-Test Split with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# 3. MODEL TRAINING & COMPARISON
# Logistic Regression
lr = LogisticRegression(class_weight="balanced", random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# Decision Tree Classifier
dt = DecisionTreeClassifier(
    max_depth=5, class_weight="balanced", random_state=42
)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("\n=== LOGISTIC REGRESSION REPORT ===")
print(classification_report(y_test, y_pred_lr))

print("\n=== DECISION TREE REPORT ===")
print(classification_report(y_test, y_pred_dt))

# 4. TOP 3 FEATURES DRIVING CHURN
feature_importances = pd.Series(dt.feature_importances_, index=X.columns)
top_3_features = feature_importances.nlargest(3)

print("\n=== TOP 3 FEATURES DRIVING CHURN ===")
for feature, importance in top_3_features.items():
    print(f"- {feature}: {importance:.4f}")

Fetching Telco Churn dataset...
Dataset Loaded Successfully! Shape: (7043, 20)


/tmp/ipykernel_3658/136774581.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)



=== LOGISTIC REGRESSION REPORT ===
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409


=== DECISION TREE REPORT ===
              precision    recall  f1-score   support

           0       0.88      0.78      0.83      1035
           1       0.54      0.71      0.61       374

    accuracy                           0.76      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.76      0.77      1409


=== TOP 3 FEATURES DRIVING CHURN ===
- Contract_Month-to-month: 0.6089
- OnlineSecurity_No: 0.1074
- MonthlyCharges: 0.0967
